<a href="https://colab.research.google.com/github/Sameer-30/Probabilistic-Damage-State-Prediction/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ----------------------------------------------------------------------
# 0. SETUP
# ----------------------------------------------------------------------
!pip install -q ngboost torch scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, log_loss,
                             mean_absolute_error, confusion_matrix)
from sklearn.calibration import calibration_curve

from ngboost import NGBClassifier
from ngboost.distns import k_categorical

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# ----------------------------------------------------------------------
# 1. LOAD + FEATURE BUILD
# ----------------------------------------------------------------------
VALUES_CSV = "train_values.csv"
LABELS_CSV = "train_labels.csv"

values = pd.read_csv(VALUES_CSV)
labels = pd.read_csv(LABELS_CSV)
df = values.merge(labels, on="building_id", how="inner")
print(f"Merged: {len(df):,} buildings")

NUM = ["age", "area_percentage", "height_percentage",
       "count_floors_pre_eq", "count_families", "geo_level_1_id"]
CAT = ["land_surface_condition", "foundation_type", "roof_type",
       "ground_floor_type", "other_floor_type", "position",
       "plan_configuration", "legal_ownership_status"]
SS  = [c for c in df.columns if c.startswith("has_superstructure")]
SEC = [c for c in df.columns if c.startswith("has_secondary_use")]

X = pd.concat([df[NUM],
               pd.get_dummies(df[CAT], drop_first=True),
               df[SS + SEC]], axis=1).astype(float)
y = df["damage_grade"].values - 1
K = 3
print(f"Feature matrix: {X.shape} | class counts: {np.bincount(y)}")

X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y, test_size=0.20, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr).astype(np.float32)
X_te_s = scaler.transform(X_te).astype(np.float32)


In [ ]:
# ======================================================================
# 2. NGBOOST  (probabilistic gradient boosting)
# ======================================================================
print("\n[NGBoost] training ...")
ngb = NGBClassifier(Dist=k_categorical(K),
                    n_estimators=500,
                    learning_rate=0.03,
                    minibatch_frac=0.5,
                    col_sample=0.7,
                    verbose=False,
                    random_state=0)
ngb.fit(X_tr_s, y_tr)
P_ngb = ngb.predict_proba(X_te_s)
pred_ngb = P_ngb.argmax(1)
ent_ngb = -(P_ngb * np.log(P_ngb + 1e-9)).sum(1)

print(f"[NGBoost] acc={accuracy_score(y_te, pred_ngb):.3f} "
      f"logloss={log_loss(y_te, P_ngb):.3f} "
      f"ordinalMAE={mean_absolute_error(y_te, pred_ngb):.3f}")

In [ ]:
# ======================================================================
# 3. PNN — CORN ORDINAL MLP  (shared architecture)
# ======================================================================

class CORN_MLP(nn.Module):
    def __init__(self, d_in, k, p_drop=0.2, hidden=(128, 64)):
        super().__init__()
        self.drop = nn.Dropout(p_drop)
        self.fc1 = nn.Linear(d_in, hidden[0])
        self.fc2 = nn.Linear(hidden[0], hidden[1])
        self.out = nn.Linear(hidden[1], k - 1)
    def forward(self, x):
        h = self.drop(F.relu(self.fc1(x)))
        h = self.drop(F.relu(self.fc2(h)))
        return self.out(h)

def corn_loss(logits, y, k):
    """Conditional BCE: task t uses only samples with y >= t."""
    loss = 0.0
    for t in range(k - 1):
        mask = (y >= t).float()
        tgt  = (y >  t).float()
        bce = F.binary_cross_entropy_with_logits(
            logits[:, t], tgt, reduction="none")
        loss = loss + (bce * mask).sum() / (mask.sum() + 1e-8)
    return loss

def corn_to_probs(logits):
    """Monotone cumulative product -> valid ordinal class probabilities."""
    s = torch.sigmoid(logits)
    cum = torch.cumprod(s, dim=1)
    p0 = 1 - cum[:, 0]
    p1 = cum[:, 0] - cum[:, 1]
    p2 = cum[:, 1]
    return torch.stack([p0, p1, p2], dim=1)

def train_one(seed, epochs=40, bs=2048, lr=2e-3, p_drop=0.2):
    torch.manual_seed(seed)
    net = CORN_MLP(X_tr_s.shape[1], K, p_drop).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-5)
    Xt = torch.tensor(X_tr_s, device=DEVICE)
    yt = torch.tensor(y_tr, device=DEVICE)
    n = len(Xt)
    net.train()
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        for s in range(0, n, bs):
            idx = perm[s:s + bs]
            opt.zero_grad()
            loss = corn_loss(net(Xt[idx]), yt[idx], K)
            loss.backward(); opt.step()
    return net

Xte_t = torch.tensor(X_te_s, device=DEVICE)

# ----------------------------------------------------------------------
# 3a. MC-DROPOUT  — keep dropout ON at test time, average T passes
# ----------------------------------------------------------------------
print("\n[PNN / MC-Dropout] training ...")
net_mc = train_one(seed=0, p_drop=0.3)
net_mc.train()
T = 50
with torch.no_grad():
    samples = torch.stack(
        [corn_to_probs(net_mc(Xte_t)) for _ in range(T)])
P_mc   = samples.mean(0).cpu().numpy()
std_mc = samples.std(0).cpu().numpy().mean(1)    # epistemic uncertainty
pred_mc = P_mc.argmax(1)
print(f"[MC-Dropout] acc={accuracy_score(y_te, pred_mc):.3f} "
      f"logloss={log_loss(y_te, P_mc):.3f} "
      f"ordinalMAE={mean_absolute_error(y_te, pred_mc):.3f}")

# ----------------------------------------------------------------------
# 3b. DEEP ENSEMBLE — M independently-trained nets, average + disagree
# ----------------------------------------------------------------------
print("\n[PNN / Deep Ensemble] training 5 members ...")
M = 5
members = []
with torch.no_grad():
    pass
ens_probs = []
for m in range(M):
    net = train_one(seed=100 + m, p_drop=0.2)
    net.eval()
    with torch.no_grad():
        ens_probs.append(corn_to_probs(net(Xte_t)).cpu().numpy())
ens_probs = np.stack(ens_probs)
P_ens   = ens_probs.mean(0)
std_ens = ens_probs.std(0).mean(1)               # epistemic uncertainty
pred_ens = P_ens.argmax(1)
print(f"[Deep Ensemble] acc={accuracy_score(y_te, pred_ens):.3f} "
      f"logloss={log_loss(y_te, P_ens):.3f} "
      f"ordinalMAE={mean_absolute_error(y_te, pred_ens):.3f}")

In [ ]:
# ======================================================================
# 4. SUMMARY TABLE
# ======================================================================
def row(name, P, pred):
    return (name, accuracy_score(y_te, pred), log_loss(y_te, P),
            mean_absolute_error(y_te, pred))
results = [row("NGBoost", P_ngb, pred_ngb),
          row("PNN MC-Dropout", P_mc, pred_mc),
          row("PNN Deep Ensemble", P_ens, pred_ens)]
print("\n================ MODEL COMPARISON ================")
print(f"{'Model':20s} {'Acc':>6s} {'LogLoss':>9s} {'OrdMAE':>8s}")
for n_, a, ll, mae in results:
    print(f"{n_:20s} {a:6.3f} {ll:9.3f} {mae:8.3f}")


In [ ]:
# ======================================================================
# 5. UQ FIGURES
# ======================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5.5))

# (a) calibration for P(DS3)
y_sev = (y_te == 2).astype(int)
for name, P, c in [("NGBoost", P_ngb, "#1b9e77"),
                   ("MC-Dropout", P_mc, "#7570b3"),
                   ("Deep Ensemble", P_ens, "#d95f02")]:
    obs, prd = calibration_curve(y_sev, P[:, 2], n_bins=10, strategy="quantile")
    axA.plot(prd, obs, "o-", color=c, lw=1.8, label=name)
axA.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
axA.set_xlabel("Predicted P(severe, DS3)")
axA.set_ylabel("Observed frequency")
axA.set_title("Calibration of severe-damage probability")
axA.legend(fontsize=9); axA.grid(alpha=0.3)

# (b) confidence (1 - normalized uncertainty) vs accuracy
def conf_acc(unc, pred, n_bins=10):
    conf = 1 - (unc - unc.min()) / (unc.max() - unc.min() + 1e-9)
    correct = (pred == y_te).astype(float)
    order = np.argsort(conf)
    xs, ys = [], []
    for b in np.array_split(order, n_bins):
        xs.append(conf[b].mean()); ys.append(correct[b].mean())
    return np.array(xs), np.array(ys)

for name, unc, pred, c in [("MC-Dropout", std_mc, pred_mc, "#7570b3"),
                           ("Deep Ensemble", std_ens, pred_ens, "#d95f02"),
                           ("NGBoost (entropy)", ent_ngb, pred_ngb, "#1b9e77")]:
    xs, ys = conf_acc(unc, pred)
    axB.plot(xs, ys, "o-", color=c, lw=1.8, label=name)
axB.set_xlabel("Model confidence  (1 - normalized uncertainty)")
axB.set_ylabel("Accuracy within bin")
axB.set_title("Does the model know when it's unsure?")
axB.legend(fontsize=9); axB.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("gorkha_uq.png", dpi=160)
plt.show()


In [ ]:
# ----------------------------------------------------------------------
# 6. EXAMPLE PREDICTIVE DISTRIBUTIONS  (most- vs least-certain buildings)
# ----------------------------------------------------------------------
order = np.argsort(std_ens)
picks = list(order[:3]) + list(order[-3:])
print("\nExample predictive distributions (Deep Ensemble):")
print(f"{'idx':>6s} {'P(DS1)':>7s} {'P(DS2)':>7s} {'P(DS3)':>7s} "
      f"{'true':>5s} {'epi_std':>8s}")
for i in picks:
    print(f"{i:6d} {P_ens[i,0]:7.2f} {P_ens[i,1]:7.2f} {P_ens[i,2]:7.2f} "
          f"{y_te[i]+1:5d} {std_ens[i]:8.3f}")

print("\nSaved: gorkha_uq.png")